# 🌍 AnyDisasterMapping — Interactive Disaster-Mapping Demo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](
https://colab.research.google.com/github/SyedRafiAhmedGIS/AnyDisasterMapping/blob/main/demo_colab.ipynb)

**Just do `Runtime ▸ Run all`** — the notebook runs top-to-bottom with **zero edits**.

## What you can upload
| Mode | What to provide | Accepted formats |
|---|---|---|
| **Single image** (segmentation) | ONE post-disaster satellite image | `.png` `.jpg` `.tif` `.tiff` `.h5` |
| **Pre/Post pair** (change detection) | TWO images: before + after the event | same as above |

> 🖼️ **Preview:** the moment a file is uploaded, this notebook renders it inline,
> prints its dimensions / bands / dtype, and tells you whether it is suitable —
> so you always *see which file should be uploaded*.

> 🛟 **No image? No problem.** A bundled synthetic sample is used automatically,
> so **Run-all never blocks or crashes**.

## How predictions are made (honest labeling)
The repo ships **no trained disaster checkpoints**, so the notebook offers 3 modes:
1. **QUICK-TRAIN** *(default)* — trains a lightweight model for a few epochs on
   synthetic data of your chosen disaster domain, then runs real inference on your
   image. Predictions are **genuine but weak** — this demonstrates the pipeline, not SOTA accuracy.
2. **CHECKPOINT** — upload your own trained `.pth` for a serious result.
3. **SYNTHETIC-DEMO** — draws clearly-labeled heuristic overlays (no training).

Every output is stamped with the mode that produced it. **Nothing is faked.**


In [ ]:
#@title ⚙️ SETUP — automatic, idempotent, restart-safe {display-mode:"form"}
import os, sys, glob, subprocess, shutil, textwrap

REPO_URL = "https://github.com/SyedRafiAhmedGIS/AnyDisasterMapping.git"  #@param {type:"string"}

def find_repo_root():
    """Locate the repo: current dir, parent, or clone it."""
    cwd = os.getcwd()
    for cand in (cwd, os.path.dirname(cwd)):
        if os.path.isfile(os.path.join(cand, "train.py")) and os.path.isdir(os.path.join(cand, "src")):
            return cand
    # maybe already cloned
    for d in glob.glob("/content/**/AnyDisasterMapping*", recursive=False) + \
             glob.glob("/content/AnyDisasterMapping*"):
        if os.path.isfile(os.path.join(d, "train.py")):
            return d
    # try github clone (silent fail → user uploads zip instead)
    try:
        r = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/content/AnyDisasterMapping"],
                           capture_output=True, text=True, timeout=300)
        if r.returncode == 0:
            return "/content/AnyDisasterMapping"
    except Exception:
        pass
    # try any uploaded zip
    zips = glob.glob("/content/*.zip") + glob.glob("*.zip")
    for z in zips:
        try:
            subprocess.run(["unzip", "-q", "-o", z, "-d", "/content/_ziprepo"], check=True)
            for root, _, files in os.walk("/content/_ziprepo"):
                if "train.py" in files:
                    return root
        except Exception:
            continue
    return None

REPO_DIR = find_repo_root()
if REPO_DIR is None:
    raise RuntimeError(
        "Could not locate the repository.\n"
        "Fix: upload AnyDisasterMapping-extended.zip to this Colab session "
        "(Files ▸ Upload) and re-run this cell.")
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print(f"✅ Repository at: {REPO_DIR}")

# ---- GPU check with friendly warning + CPU fallback flag ----
import torch
GPU = torch.cuda.is_available()
DEVICE = torch.device("cuda" if GPU else "cpu")
if GPU:
    print(f"✅ GPU detected: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ No GPU — running in CPU fallback mode. "
          "QUICK-TRAIN will be slower; consider Runtime ▸ Change runtime type ▸ T4 GPU.")

# ---- pip install (pinned, quiet, idempotent) ----
MARKER = "/content/.adm_deps_installed"
if not os.path.exists(MARKER):
    print("📦 Installing pinned dependencies (first run only, ~2-4 min)...")
    # Reuse Colab's preinstalled torch — never reinstall torch on Colab.
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--no-deps", "-e", "."],
                   check=True)
    deps = ["numpy==1.26.4", "albumentations==1.4.24", "pyyaml", "tqdm", "imageio",
            "tifffile", "imagecodecs", "scikit-image", "rasterio", "termcolor",
            "prettytable", "safetensors", "yacs", "timm", "h5py", "hdf5plugin",
            "scikit-learn", "einops", "fvcore", "tensorboardX", "pandas", "Pillow",
            "opencv-python-headless", "shapely", "ipywidgets"]
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *deps], check=True)
    open(MARKER, "w").write("ok")
    print("✅ Dependencies installed.")
else:
    print("✅ Dependencies already installed (idempotent skip).")

# ---- folders ----
for d in ["/content/uploads", "/content/results/predicted_masks",
          "/content/results/overlay_images", "/content/results/comparison_grids",
          "/content/results/plots", "/content/checkpoints"]:
    os.makedirs(d, exist_ok=True)
print("✅ Folders ready.")

import matplotlib.pyplot as plt
import numpy as np
print(f"\nRuntime summary → torch {torch.__version__} | device {DEVICE} | python {sys.version.split()[0]}")


In [ ]:
#@title ☁️ MOUNT GOOGLE DRIVE — persistent uploads, checkpoints & results {display-mode:"form"}
MOUNT_DRIVE = True  #@param {type:"boolean"}
DRIVE_PROJECT = "AnyDisasterMapping"  #@param {type:"string"}

# Shared notebook core lives in tools/colab_kit.py (single source of truth).
def _ensure_colab_kit():
    import os as _os, sys as _sys, tempfile as _tf, urllib.request as _uq
    _d = _os.path.abspath(_os.getcwd())
    for _ in range(8):
        _p = _os.path.join(_d, "tools")
        if _os.path.isfile(_os.path.join(_p, "colab_kit.py")):
            if _p not in _sys.path:
                _sys.path.insert(0, _p)
            return "local", _p
        _nd = _os.path.dirname(_d)
        if _nd == _d:
            break
        _d = _nd
    _dest = _os.path.join(_tf.gettempdir(), "adm_colab_kit")
    _os.makedirs(_dest, exist_ok=True)
    _fp = _os.path.join(_dest, "colab_kit.py")
    if not _os.path.isfile(_fp):
        for _rev in ("v5.0.0", "main"):
            _url = ("https://raw.githubusercontent.com/SyedRafiAhmedGIS/"
                    f"AnyDisasterMapping/{_rev}/tools/colab_kit.py")
            try:
                _uq.urlretrieve(_url, _fp)
                if _rev != "v5.0.0":
                    print("⚠️ colab_kit @ v5.0.0 not published yet — fetched 'main'; "
                          "pin activates once the tag is pushed.")
                else:
                    print(f"📥 fetched tools/colab_kit.py @ {_rev}")
                break
            except Exception:
                continue
        else:
            raise RuntimeError("colab_kit.py not found locally and fetch failed — "
                               "run this notebook from the repository checkout")
    if _dest not in _sys.path:
        _sys.path.insert(0, _dest)
    return "fetched", _dest

_ensure_colab_kit()
import colab_kit

DRIVE_ROOT, IN_COLAB = colab_kit.mount_and_prepare(DRIVE_PROJECT, MOUNT_DRIVE)

In [ ]:
#@title 📤 UPLOAD — interactive buttons, inline preview, never blocks {display-mode:"form"}
import os, glob, threading, traceback
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# ---------- state ----------
STATE = {"mode": None,            # "seg" or "cd"
         "files": [],             # list of local paths [pre?] or [single]
         "domain": "flood",
         "used_fallback": False,
         "meta": {}}

DOMAINS_SEG = ["flood", "landslide", "wildfire", "oil_spill", "avalanche",
               "volcano", "glof", "dust_storm", "smoke_plume", "locust"]
DOMAINS_CD = ["flood", "tsunami", "hurricane", "oil_spill", "locust", "glof"]

mode_sel = widgets.ToggleButtons(options=[("🖼️ Single image (segmentation)", "seg"),
                                          ("🆚 Pre/Post pair (change detection)", "cd")],
                                 value="seg", description="Mode:")
domain_sel = widgets.Dropdown(options=DOMAINS_SEG, value="flood", description="Domain:")
gsd_box = widgets.FloatText(value=10.0, description="GSD (m/px):",
                            tooltip="Ground sample distance for km² stats; leave 10 if unknown")
upload_btn = widgets.Button(description="⬆️ Upload image(s)", button_style="primary",
                            icon="upload")
fallback_btn = widgets.Button(description="🛟 No image? Use bundled synthetic sample",
                              button_style="warning", icon="picture")
out_up = widgets.Output()

def _sync_domains(*_):
    domain_sel.options = DOMAINS_CD if mode_sel.value == "cd" else DOMAINS_SEG
mode_sel.observe(_sync_domains, names="value")

def _read_any(path):
    """Read image to float32 HWC + metadata dict."""
    ext = os.path.splitext(path)[1].lower()
    if ext in (".tif", ".tiff"):
        import tifffile
        arr = np.asarray(tifffile.imread(path), dtype=np.float32)
        meta = {}
        try:
            import rasterio
            with rasterio.open(path) as src:
                meta = {"crs": str(src.crs), "gsd": src.res[0], "bounds": str(src.bounds)}
        except Exception:
            pass
    elif ext in (".h5", ".hdf5"):
        import h5py
        with h5py.File(path, "r") as f:
            arr = np.asarray(f[next(iter(f.keys()))], dtype=np.float32)
        meta = {}
    else:
        import imageio.v2 as imageio
        arr = np.asarray(imageio.imread(path), dtype=np.float32)
        meta = {}
    return arr, meta

def _show_preview(paths, mode):
    """Inline preview + suitability verdict — WHICH FILE SHOULD BE UPLOADED."""
    with out_up:
        clear_output(wait=True)
        n = len(paths)
        fig, axes = plt.subplots(1, n, figsize=(5 * n, 4.6))
        if n == 1: axes = [axes]
        for ax, p in zip(axes, paths):
            arr, meta = _read_any(p)
            if arr.ndim == 3 and arr.shape[0] <= 8 and arr.shape[0] < arr.shape[1]:
                arr = np.transpose(arr, (1, 2, 0))  # CHW → HWC
            img2d = arr[..., 0] if arr.ndim == 3 else arr
            vmax = np.percentile(img2d, 99) if img2d.max() > 4 else img2d.max()
            ax.imshow(img2d, cmap="gray" if arr.ndim == 2 else None,
                      vmin=0, vmax=max(vmax, 1e-6))
            band_txt = (f"{arr.shape[-1]} bands" if arr.ndim == 3 else "grayscale")
            ax.set_title(f"{os.path.basename(p)}\n{arr.shape[0]}×{arr.shape[1]} · {band_txt} · {arr.dtype}")
            ax.axis("off")
            STATE["meta"] = {"shape": arr.shape, "bands": (arr.shape[-1] if arr.ndim == 3 else 1),
                             "dtype": str(arr.dtype), **meta}
        verdict = []
        m = STATE["meta"]
        verdict.append(f"✓ readable {m['dtype']} image, {m['bands']} band(s)")
        if m["bands"] == 1:
            verdict.append("ℹ 1-band input → will be replicated to 3 channels")
        if m["bands"] > 8:
            verdict.append("⚠ >8 bands → only first 8 kept")
        if mode == "cd" and n != 2:
            verdict.append("✗ change-detection mode needs EXACTLY 2 files (pre & post)")
        ok = not any(v.startswith("✗") for v in verdict)
        plt.show()
        print("Preview verdict:", " | ".join(verdict))
        print("Mode selected:", "SEGMENTATION (single image)" if mode == "seg"
              else "CHANGE DETECTION (pre+post pair)")
        if not ok:
            print("⚠ Adjust your upload or switch mode above.")

def _on_upload(_):
    from google.colab import files as colab_files  # noqa
    print("Select 1 file (seg) or 2 files (pre & post, cd)…")
    os.makedirs("/content/uploads", exist_ok=True)
    up = colab_files.upload()
    paths = []
    for name in up:
        dst = os.path.join("/content/uploads", name)
        open(dst, "wb").write(up[name])
        paths.append(dst)
    STATE["mode"] = mode_sel.value
    STATE["domain"] = domain_sel.value
    STATE["files"] = sorted(paths)
    STATE["used_fallback"] = False
    if STATE["mode"] == "cd" and len(paths) == 2:
        print("Assume alphabetical order = pre,post  (rename files 0_pre… / 1_post… to control)")
    _show_preview(STATE["files"], STATE["mode"])

def _make_synthetic_sample():
    """Bundled synthetic fallback sample so Run-all NEVER blocks."""
    rng = np.random.default_rng(7)
    def synth(colorshift=0.0):
        img = rng.normal(0.45, 0.15, (128, 128, 3)) + colorshift
        yy, xx = np.mgrid[0:128, 0:128]
        img[..., 0] += 0.25 * (((yy - 70) / 25) ** 2 + ((xx - 50) / 30) ** 2 <= 1)
        return np.clip(img, 0, 1)
    os.makedirs("/content/uploads", exist_ok=True)
    p1 = "/content/uploads/synthetic_sample.tif"
    import tifffile
    tifffile.imwrite(p1, synth(0.0).astype(np.float32))
    if mode_sel.value == "cd":
        p2 = "/content/uploads/synthetic_sample_post.tif"
        tifffile.imwrite(p2, synth(0.15).astype(np.float32))
        paths = sorted([p1, p2])
    else:
        paths = [p1]
    STATE.update(mode=mode_sel.value, domain=domain_sel.value,
                 files=paths, used_fallback=True)
    print("🛟 Using BUNDLED SYNTHETIC SAMPLE (no user image provided).")
    _show_preview(paths, STATE["mode"])

def _on_fallback(_):
    try:
        _make_synthetic_sample()
    except Exception:
        traceback.print_exc()

upload_btn.on_click(_on_upload)
fallback_btn.on_click(_on_fallback)

box = widgets.VBox([mode_sel, domain_sel, gsd_box,
                    widgets.HBox([upload_btn, fallback_btn]), out_up])
display(box)

# AUTO-FALLBACK for Run-all: if no upload happened, run non-blocking, continue.
if not STATE["files"]:
    _make_synthetic_sample()
    print("▶ Run-all continuation: synthetic sample auto-selected. "
          "Upload your own image later and re-run from this cell.")

print(f"\nSTATE → mode={STATE['mode']}  domain={STATE['domain']}  "
      f"files={[os.path.basename(f) for f in STATE['files']]}  "
      f"fallback={STATE['used_fallback']}")


In [ ]:
#@title 🧠 MODEL — 3 clearly-labeled modes (never fakes results) {display-mode:"form"}
import os, sys, glob, math, traceback
import numpy as np
import torch
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

MODE_SEL = widgets.ToggleButtons(
    options=[("⚡ QUICK-TRAIN (recommended)", "quicktrain"),
             ("💾 CHECKPOINT (your .pth)", "checkpoint"),
             ("🎨 SYNTHETIC-DEMO (no training)", "synthetic")],
    value="quicktrain", description="Mode:")
EPOCH_SLIDER = widgets.IntSlider(value=3, min=1, max=10, step=1,
                                 description="Epochs:", continuous_update=False)
ckpt_btn = widgets.Button(description="⬆️ Upload .pth checkpoint", icon="upload")
out_model = widgets.Output()

CKPT_STATE = {"path": None}

def _on_ckpt(_):
    from google.colab import files as colab_files
    os.makedirs("/content/checkpoints", exist_ok=True)
    up = colab_files.upload()
    for name, data in up.items():
        CKPT_STATE["path"] = os.path.join("/content/checkpoints", name)
        open(CKPT_STATE["path"], "wb").write(data)
    with out_model:
        print(f"💾 Checkpoint saved → {CKPT_STATE['path']}")
ckpt_btn.on_click(_on_ckpt)

display(widgets.VBox([MODE_SEL, EPOCH_SLIDER, widgets.HBox([ckpt_btn]), out_model]))

# ============================================================
# Shared helpers
# ============================================================
from src.core.config import Config
from src.core.registry import model_libs, dataset_libs

CONFIG_BY_DOMAIN = {
    "flood":      "configs/flood/cau/seg_unet.yaml",
    "landslide":  "configs/landslide/l4s/seg_unet.yaml",
    "wildfire":   "configs/smoke_plume/smoke_modis/seg_unet.yaml",
    "oil_spill":  "configs/oil_spill/oilspill_sar/seg_unet.yaml",
    "avalanche":  "configs/avalanche/avalanche_slf/seg_unet.yaml",
    "volcano":    "configs/volcano/volcano_pyraws/seg_unet.yaml",
    "glof":       "configs/glof/icimod/seg_unet.yaml",
    "dust_storm": "configs/dust_storm/modis_dust/seg_unet.yaml",
    "smoke_plume":"configs/smoke_plume/smoke_modis/seg_unet.yaml",
    "tsunami":    "configs/tsunami/xbd_tsunami/cd_bit.yaml",
    "hurricane":  "configs/hurricane/s1_surge/cd_bit.yaml",
    "locust":     "configs/locust/fao_s2/seg_unet.yaml",
}
CONFIG_BY_DOMAIN["wildfire"] = CONFIG_BY_DOMAIN["smoke_plume"]

CLASS_NAMES = {
    "flood": ["background", "flood water"],
    "landslide": ["background", "landslide"],
    "oil_spill": ["sea", "oil slick", "look-alike"],
    "avalanche": ["background", "avalanche debris"],
    "volcano": ["background", "lava flow", "vent", "fumarole"],
    "tsunami": ["no-change", "inundated"],
    "hurricane": ["no-change", "surge inundation"],
    "glof": ["land", "glacial lake"],
    "dust_storm": ["clear", "dust plume", "cloud"],
    "smoke_plume": ["clear", "smoke", "cloud"],
    "locust": ["background", "affected cropland"],
}

def make_dummy_for_domain(domain, out_dir, config):
    """Generate the tiny dummy dataset matching the config's channel budget.

    Layout families:
      single-IMG  : avalanche, volcano, dust_storm, smoke_plume, wildfire
      PRE/POST/GT : flood, landslide, oil_spill, tsunami, hurricane, locust
      glof (T1/T2/GT_CD/GT_T1/GT_T2) : glof
    Per-date channels are extracted TASK-AWARE so the model always receives
    the exact tensor shape it was built for:
      seg configs     -> model.kwargs.in_channels = stacked channels
      cd BIT configs  -> model.kwargs.input_nc    = per-date channels
      cd ChangeOS     -> model.kwargs.in_channels = 2 x per-date channels
    """
    import importlib.util
    from pathlib import Path
    spec = importlib.util.spec_from_file_location(
        "dgen", "scripts/data_prep/_dummy_common.py")
    dgen = importlib.util.module_from_spec(spec); spec.loader.exec_module(dgen)
    out = Path(out_dir)
    n_cls = int(config.get("num_classes", 2))
    kw = config.get("model", {}).get("kwargs", {})
    task = str(config.get("task", "seg")).lower()
    if task == "cd":
        per_date = int(kw.get("input_nc") or int(kw.get("in_channels", 6)) // 2 or 3)
        stack = per_date * 2
    else:
        stack = int(kw.get("in_channels", 3) or 3)
        per_date = None
    ids = dgen.default_ids()
    if domain == "glof":
        dgen.gen_glof(out, 64, ids)
    elif domain in ("avalanche", "volcano", "dust_storm", "smoke_plume", "wildfire"):
        ch = stack  # single-image datasets use all channels in one image
        dgen.gen_single_domain(out, 64, ch, n_cls, ids)
    elif task == "cd":
        dgen.gen_pair_domain(out, 64, per_date, max(n_cls, 2), ids, seg=False)
    else:  # seg PRE/POST/GT pair domains (stacked channels, odd-safe)
        dgen.gen_pair_domain(out, 64, (stack + 1) // 2, n_cls, ids,
                             ch_post=max(stack // 2, 1))

def quick_train(domain, epochs):
    """Mode 1: train lightweight model on dummy data, return (model, config, loss_log)."""
    cfg_path = CONFIG_BY_DOMAIN[domain]
    config = Config.from_yaml(cfg_path)
    dummy_dir = f"/content/dummy_{domain}"
    os.makedirs(dummy_dir, exist_ok=True)
    make_dummy_for_domain(domain, dummy_dir, config)
    for split in ("train", "val", "test"):
        config._data["dataset"].setdefault(split, {}).update(
            {"dataset_path": dummy_dir, "data_list_path": f"{dummy_dir}/{split}.txt"})
    config._data["training"].update({"num_epochs": epochs, "train_batch_size": 2,
                                     "eval_batch_size": 1, "num_workers": 0})
    config._data["scheduler"] = None
    # dummy tiles are 64x64; shrink EVERY crop transform in EVERY split
    # (train RandomCrop and val CenterCrop both raise CropSizeError on
    #  64px tiles; mirrors the smoke-test override, generalized)
    for _split in ("train", "val", "test"):
        _tf = config._data.get("augmentation", {}).get(_split, {})
        if not isinstance(_tf, dict):
            continue
        for _name, _params in _tf.items():
            if (isinstance(_params, dict) and "crop" in _name.lower()
                    and "height" in _params and "width" in _params):
                _params["height"] = 64
                _params["width"] = 64
    config._data["output_dir"] = "/content/results/quicktrain"
    from src.core.trainer import Trainer
    trainer = Trainer(config)
    # capture loss curve
    loss_log = []
    orig_step = trainer._train_step
    def step_wrap(batch):
        loss = orig_step(batch)
        loss_log.append(loss)
        return loss
    trainer._train_step = step_wrap
    trainer.train()
    return trainer, config, loss_log

def load_model_from_config(cfg_path, ckpt_path=None):
    """Build model from config and optionally load a checkpoint."""
    config = Config.from_yaml(cfg_path)
    factory = model_libs[config.get("model", {}).get("name")]
    model = factory(**config.get("model", {}).get("kwargs", {}))
    if ckpt_path:
        sd = torch.load(ckpt_path, map_location="cpu", weights_only=False)
        sd = sd.get("state_dict", sd)
        model.load_state_dict(sd, strict=True)
    model.eval()
    return model, config

def preprocess_uploaded(path, config, target_size=128, channels=None):
    """Upload → normalized (1,C,H,W) tensor using the config's Normalize stats.

    channels: target band count. None → auto from config (input_nc for
    per-date cd models, else in_channels). The array is repeated/trimmed
    to exactly that many bands so the model gets its expected shape.
    """
    arr, meta = _read_any(path)
    if arr.ndim == 3 and arr.shape[0] <= 8 and arr.shape[0] < arr.shape[1]:
        arr = np.transpose(arr, (1, 2, 0))
    if arr.ndim == 2:
        arr = arr[:, :, None]
    if arr.shape[-1] > 8:
        arr = arr[..., :8]
    if arr.shape[-1] < 3 and arr.shape[-1] != 1:
        arr = np.repeat(arr, 3, axis=-1)[..., :3]
    # match the model's expected channel budget (mirrors training input)
    if channels is None:
        _kw = {}
        try:
            _kw = config.get("model", {}).get("kwargs", {}) or {}
        except Exception:
            _kw = {}
        if _kw.get("input_nc"):
            channels = int(_kw["input_nc"])
        elif _kw.get("in_channels"):
            channels = int(_kw["in_channels"])
    if channels and arr.shape[-1] != int(channels):
        _c = int(channels)
        if arr.shape[-1] < _c:
            _reps = int(np.ceil(_c / arr.shape[-1]))
            arr = np.repeat(arr, _reps, axis=-1)[..., :_c]
        else:
            arr = arr[..., :_c]
    # normalize per config (robust fallback: min-max)
    norm = None
    try:
        val_aug = config.get("augmentation", {}).get("val", {})
        ncfg = val_aug.get("Normalize", {})
        mean = np.array(ncfg.get("mean", [0.5] * arr.shape[-1]), dtype=np.float32)
        std = np.array(ncfg.get("std", [0.5] * arr.shape[-1]), dtype=np.float32)
        if arr.shape[-1] == mean.shape[0]:
            mx = float(ncfg.get("max_pixel_value", 1.0))
            if mx > 2.0 and float(arr.max()) <= 2.0:
                arr = arr * mx  # 0-1 float input vs 0-255 stats
            arr = (arr.astype(np.float32) / mx - mean) / std
        else:
            arr = (arr - arr.min()) / max(arr.max() - arr.min(), 1e-6)
    except Exception:
        arr = (arr - arr.min()) / max(arr.max() - arr.min(), 1e-6)
    # resize to multiple of 32 (safety for ViT heads)
    h, w = arr.shape[:2]
    hs = max(32, int(math.ceil(h / 32) * 32)) if max(h, w) > 32 else 64
    ws = max(32, int(math.ceil(w / 32) * 32)) if max(h, w) > 32 else 64
    try:
        import cv2
        arr = cv2.resize(arr.astype(np.float32), (ws, hs))
    except Exception:
        from skimage.transform import resize
        arr = resize(arr.astype(np.float32), (hs, ws, arr.shape[-1]), order=1)
    chw = np.ascontiguousarray(arr.transpose(2, 0, 1))[None, ...]
    return torch.from_numpy(chw).float()

@torch.no_grad()
def _forward(model, tensor, tensor_pre=None):
    """Call the model with the right input arity.

    seg/scd single-tensor models → model(x); siamese cd models
    (BIT, SiamCRNN, DSIFN, ChangeMamba…) → model(pre, post);
    single-tensor cd models (ChangeOS) → model(cat([pre, post], dim=1)).
    """
    dev = next(model.parameters()).device
    if tensor_pre is not None:
        try:
            return model(tensor_pre.to(dev), tensor.to(dev))
        except TypeError:
            return model(torch.cat([tensor_pre, tensor], dim=1).to(dev))
    return model(tensor.to(dev))

@torch.no_grad()
def predict(model, tensor, tensor_pre=None):
    """Run model → argmax mask (H, W) on CPU numpy."""
    out = _forward(model, tensor, tensor_pre)
    if hasattr(out, "logits"):
        out = out.logits
    if isinstance(out, (tuple, list)):
        out = out[-1] if len(out) == 2 else out[0]
    pred = torch.argmax(out, dim=1)[0].cpu().numpy()
    return pred

PALETTE = np.array([[0, 0, 0], [56, 116, 255], [255, 96, 0], [255, 216, 0],
                    [0, 200, 120], [200, 0, 200], [128, 128, 128]], dtype=np.uint8)

def colorize(mask):
    Pal = PALETTE
    out = np.zeros((*mask.shape, 3), dtype=np.uint8)
    for c in np.unique(mask):
        if c < len(Pal):
            out[mask == c] = Pal[c]
    return out

print("✅ Model helpers ready.")


In [ ]:
#@title ▶️ RUN the selected mode + show RESULTS dashboard {display-mode:"form"}
import os, json, math, datetime, traceback
for _d in ("results/predicted_masks", "results/overlay_images",
           "results/comparison_grids", "results/plots"):
    os.makedirs(f"/content/{_d}", exist_ok=True)
import numpy as np
import torch
import matplotlib.pyplot as plt
from IPython.display import display

RESULTS = {"mode": None, "pred": None, "pre": None, "post": None,
           "loss_log": None, "class_names": None, "domain": STATE["domain"]}
t_pre = None  # pre-event tensor (cd mode)

domain = STATE["domain"]
class_names = CLASS_NAMES.get(domain, ["background", "target"])
n_classes = max(len(class_names), 2)

# ============================================================
# MODE EXECUTION — the mode that produces the output is PRINTED ALWAYS
# ============================================================
try:
    if MODE_SEL.value == "quicktrain":
        print(f"🏷️ OUTPUT MODE = QUICK-TRAIN | domain={domain} | epochs={EPOCH_SLIDER.value}")
        print("   Predictions are GENUINE but WEAK (trained minutes ago on synthetic data).")
        trainer, config, loss_log = quick_train(domain, EPOCH_SLIDER.value)
        model = trainer.model
        RESULTS.update(mode="QUICK-TRAIN", loss_log=loss_log)

    elif MODE_SEL.value == "checkpoint":
        print("🏷️ OUTPUT MODE = CHECKPOINT (user-supplied weights)")
        if not CKPT_STATE["path"]:
            print("   No checkpoint uploaded → falling back to QUICK-TRAIN.")
            trainer, config, loss_log = quick_train(domain, EPOCH_SLIDER.value)
            model = trainer.model
            RESULTS.update(mode="QUICK-TRAIN (checkpoint fallback)", loss_log=loss_log)
        else:
            model, config = load_model_from_config(CONFIG_BY_DOMAIN[domain], CKPT_STATE["path"])
            RESULTS.update(mode="CHECKPOINT")
        print("   ⚠ Checkpoint must match this architecture; mismatches will error honestly.")

    else:
        print("🏷️ OUTPUT MODE = SYNTHETIC-DEMO | clearly-labeled heuristic overlay — NOT a real prediction")
        model = None
        RESULTS.update(mode="SYNTHETIC-DEMO")
except Exception:
    traceback.print_exc()
    print("\n🛟 Training/inference failed → falling back to SYNTHETIC-DEMO overlay.")
    model = None
    RESULTS.update(mode="SYNTHETIC-DEMO (error fallback)")

# ============================================================
# INFERENCE on the uploaded (or synthetic) image
# ============================================================
def heuristic_demo(img2d_or_rgb):
    """Clearly-labeled heuristic placeholder: bright/low-sat regions = 'target'."""
    rgb = img2d_or_rgb if img2d_or_rgb.ndim == 3 else np.stack([img2d_or_rgb] * 3, -1)
    gray = rgb.mean(-1)
    sat = rgb.max(-1) - rgb.min(-1)
    pred = ((gray > np.percentile(gray, 70)) & (sat < np.percentile(sat, 80))).astype(np.int64)
    return pred

img_path = STATE["files"][-1]
pre_path = STATE["files"][0] if (STATE["mode"] == "cd" and len(STATE["files"]) == 2) else None

raw, _ = _read_any(img_path)
if raw.ndim == 3 and raw.shape[0] <= 8 and raw.shape[0] < raw.shape[1]:
    raw = np.transpose(raw, (1, 2, 0))
disp_img = raw if raw.ndim == 3 else np.stack([raw] * 3, -1)
disp_img = np.clip(disp_img / max(disp_img.max(), 1e-6), 0, 1)

if pre_path:
    raw_pre, _ = _read_any(pre_path)
    if raw_pre.ndim == 3 and raw_pre.shape[0] <= 8 and raw_pre.shape[0] < raw_pre.shape[1]:
        raw_pre = np.transpose(raw_pre, (1, 2, 0))
    disp_pre = raw_pre if raw_pre.ndim == 3 else np.stack([raw_pre] * 3, -1)
    disp_pre = np.clip(disp_pre / max(disp_pre.max(), 1e-6), 0, 1)
else:
    disp_pre = None

if model is not None:
    try:
        _task = str(config.get("task", "seg")).lower()
        _kw = config.get("model", {}).get("kwargs", {}) or {}
        if _task == "cd":
            _per_date = (int(_kw["input_nc"]) if _kw.get("input_nc")
                         else (int(_kw.get("in_channels", 0) or 0) // 2) or None)
        else:
            _per_date = None  # single-tensor tasks: auto budget = in_channels
        t = preprocess_uploaded(img_path, config, channels=_per_date)
        t_pre = (preprocess_uploaded(pre_path, config, channels=_per_date)
                 if pre_path else None)
        pred = predict(model, t, t_pre)
        if pred.shape != disp_img.shape[:2]:
            import cv2
            pred = cv2.resize(pred, (disp_img.shape[1], disp_img.shape[0]),
                              interpolation=cv2.INTER_NEAREST)
        RESULTS["pred"] = pred
    except Exception:
        traceback.print_exc()
        RESULTS["pred"] = None

if RESULTS["pred"] is None:
    RESULTS["pred"] = heuristic_demo(disp_img)
    if not RESULTS["mode"].startswith("SYNTHETIC"):
        RESULTS["mode"] += " + heuristic fallback"

RESULTS.update(class_names=class_names)
print(f"\n🏷️ FINAL OUTPUT PRODUCED BY: {RESULTS['mode']}  (domain={domain})")

# ============================================================
# DASHBOARD
# ============================================================
pred = RESULTS["pred"]
np.save("/content/results/predicted_masks/pred_mask.npy", pred)
plt.imsave("/content/results/predicted_masks/pred_mask.png", colorize(pred))
GSD = gsd_box.value if gsd_box.value and gsd_box.value > 0 else None
px_km2 = (GSD / 1000.0) ** 2 if GSD else None

overlay = disp_img.copy()
mask_rgb = colorize(pred) / 255.0
blend = 0.55 * disp_img + 0.45 * mask_rgb
blend[pred == 0] = disp_img[pred == 0]

# ---- stats table ----
stats_rows = []
total_px = pred.size
for c in range(n_classes):
    n = int((pred == c).sum())
    pct = 100.0 * n / total_px
    area = n * px_km2 if px_km2 else float("nan")
    stats_rows.append((class_names[c] if c < len(class_names) else f"class {c}",
                       n, pct, area))

fig, axes = plt.subplots(1, 3 if STATE["mode"] == "cd" and disp_pre is not None else 2,
                         figsize=(15, 4.8))
axes[0].imshow(disp_img); axes[0].set_title("Input image"); axes[0].axis("off")
axes[1].imshow(np.clip(blend, 0, 1)); axes[1].set_title(f"Overlay — mode: {RESULTS['mode']}")
axes[1].axis("off")
if axes.shape[0] == 3 and disp_pre is not None:
    diff = (pred > 0).astype(np.float32) - (heuristic_demo(disp_pre) > 0)
    axes[2].imshow(disp_pre); axes[2].set_title("Pre-event image"); axes[2].axis("off")
plt.tight_layout()
plt.savefig("/content/results/overlay_images/overlay_main.png", dpi=110, bbox_inches="tight")
plt.show()

# legend
import matplotlib.patches as mpatches
fig, ax = plt.subplots(figsize=(6, 1.2 + 0.3 * n_classes))
ax.axis("off")
handles = [mpatches.Patch(color=PALETTE[c] / 255.0,
                          label=f"{class_names[c] if c < len(class_names) else c}") for c in range(n_classes)]
ax.legend(handles=handles, loc="center", ncol=min(n_classes, 4), frameon=False)
ax.set_title("Class legend")
plt.savefig("/content/results/overlay_images/legend.png", dpi=110, bbox_inches="tight")
plt.show()

# ---- cd comparison grid ----
if STATE["mode"] == "cd" and disp_pre is not None:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.8))
    axes[0].imshow(disp_pre); axes[0].set_title("PRE"); axes[0].axis("off")
    axes[1].imshow(disp_img); axes[1].set_title("POST"); axes[1].axis("off")
    change = (pred > 0).astype(np.float32)
    axes[2].imshow(disp_img); axes[2].imshow(np.ma.masked_where(change == 0, change),
                                             cmap="autumn_r", alpha=0.55)
    axes[2].set_title("Change highlight (mode: " + RESULTS["mode"] + ")"); axes[2].axis("off")
    plt.tight_layout()
    plt.savefig("/content/results/comparison_grids/pre_post_change.png", dpi=110, bbox_inches="tight")
    plt.show()

# ---- plots: class distribution, pie, loss curve, confidence hist ----
names = [r[0] for r in stats_rows]; counts = [r[1] for r in stats_rows]
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].bar(range(len(counts)), [100 * c / total_px for c in counts],
            color=[PALETTE[i] / 255 for i in range(len(counts))])
axes[0].set_xticks(range(len(names))); axes[0].set_xticklabels(names, rotation=20, ha="right")
axes[0].set_ylabel("% of scene"); axes[0].set_title("Class distribution")
axes[1].pie([max(c, 1) for c in counts], labels=names,
            colors=[PALETTE[i] / 255 for i in range(len(counts))],
            autopct="%1.1f%%", textprops={"fontsize": 8})
axes[1].set_title("Scene composition")
if RESULTS["loss_log"]:
    axes[2].plot(RESULTS["loss_log"]); axes[2].set_xlabel("iteration")
    axes[2].set_ylabel("loss"); axes[2].set_title("Training loss curve (quick-train)")
else:
    axes[2].text(0.5, 0.5, "no loss curve\n(mode has no training)",
                 ha="center", va="center"); axes[2].axis("off")
plt.tight_layout()
plt.savefig("/content/results/plots/class_pie_loss.png", dpi=110, bbox_inches="tight")
plt.show()

conf_hist = None
if model is not None:
    try:
        with torch.no_grad():
            _kw = config.get("model", {}).get("kwargs", {}) or {}
            _pd = int(_kw["input_nc"]) if _kw.get("input_nc") else None
            t = preprocess_uploaded(img_path, config, channels=_pd)
            out = _forward(model, t, t_pre)
            if hasattr(out, "logits"): out = out.logits
            if isinstance(out, (tuple, list)): out = out[-1] if len(out) == 2 else out[0]
            conf = torch.softmax(out, dim=1).max(dim=1)[0].cpu().numpy().ravel()
        plt.figure(figsize=(6, 3.2))
        plt.hist(conf, bins=40, color="#3874ff", alpha=0.85)
        plt.xlabel("max softmax confidence"); plt.ylabel("pixels")
        plt.title("Prediction confidence histogram")
        plt.tight_layout()
        plt.savefig("/content/results/plots/confidence_hist.png", dpi=110, bbox_inches="tight")
        plt.show()
    except Exception:
        pass

# ---- stats CSV/JSON ----
import csv
with open("/content/results/stats.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["class", "pixels", "percent_of_scene", "area_km2"])
    for row in stats_rows:
        w.writerow([row[0], row[1], f"{row[2]:.3f}",
                    f"{row[3]:.4f}" if row[3] == row[3] else "unknown_GSD"])
with open("/content/results/stats.json", "w") as f:
    json.dump({"mode": RESULTS["mode"], "domain": domain, "gsd_m": GSD,
               "stats": [{"class": r[0], "pixels": r[1], "percent": r[2],
                          "area_km2": (r[3] if r[3] == r[3] else None)} for r in stats_rows]},
              f, indent=2)

# ---- plain-language REPORT ----
top = sorted(stats_rows, key=lambda r: -r[1])
lines = [f"REPORT — generated {datetime.datetime.now():%Y-%m-%d %H:%M}",
         f"Output mode: {RESULTS['mode']} (domain: {domain})",
         f"Scene: {total_px:,} pixels" + (f", ≈{total_px * px_km2:.2f} km² at GSD {GSD} m" if px_km2 else " (GSD unknown → no km²)"),
         ""]
for name, n, pct, area in top[:3]:
    a_txt = f" ≈ {area:.2f} km²" if area == area else ""
    lines.append(f"• {name}: {pct:.1f}% of the scene ({n:,} px){a_txt}")
if top[0][2] > 60:
    lines.append("The dominant class covers most of the scene — for disaster mapping you may "
                 "want to inspect whether the model is over-predicting the background class.")
else:
    lines.append("Class proportions look varied; inspect the overlay for spatial coherence.")
lines.append("Caveat: " + ("predictions come from a briefly-trained model on synthetic data — "
             "treat as a pipeline demonstration." if "QUICK-TRAIN" in RESULTS["mode"]
             else "verify predictions against your own labels before operational use."))
report = "\n".join(lines)
with open("/content/results/report.txt", "w") as f:
    f.write(report)
print("\n" + "=" * 62 + "\n" + report + "\n" + "=" * 62)


In [ ]:
#@title 📦 EXPORT — results_analysis_<timestamp>.zip + manifest + auto-download {display-mode:"form"}
import os, json, zipfile, glob, datetime, shutil

ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
zip_path = f"/content/results_analysis_{ts}.zip"

include_patterns = ["/content/results/**/*", "/content/uploads/*", "/content/dummy_*/*"]
manifest = []
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for pat in include_patterns:
        for p in glob.glob(pat, recursive=True):
            if os.path.isfile(p):
                zf.write(p, arcname=os.path.relpath(p, "/content"))
                manifest.append(os.path.relpath(p, "/content"))

# write manifest inside the zip too
mpath = "/content/results/manifest.txt"
with open(mpath, "w") as f:
    f.write("\n".join(sorted(manifest)))
with zipfile.ZipFile(zip_path, "a") as zf:
    zf.write(mpath, arcname="results/manifest.txt")

size_mb = os.path.getsize(zip_path) / 1e6
print(f"✅ Packaged {len(manifest)} files → {zip_path} ({size_mb:.1f} MB)")
print("\nFILE MANIFEST")
print("-" * 50)
for m in sorted(manifest)[:60]:
    print("  ", m)
if len(manifest) > 60:
    print(f"   … and {len(manifest) - 60} more (full list inside manifest.txt)")

try:
    from google.colab import files
    files.download(zip_path)
    print("⬇️ Download started.")
except Exception as e:
    print(f"(auto-download unavailable here: {e}) — the zip is saved at {zip_path}")
